# Prepare authorized data

Parse locally held i2b2/VA files and create an ID-only split manifest. All text-bearing outputs remain under Git-ignored data directories.

In [ ]:
from pathlib import Path

from clinical_re_bilingual.data.i2b2 import (
    I2B2Layout,
    parse_i2b2_directory,
    read_relation_examples_tsv,
    write_tsv,
)
from clinical_re_bilingual.data.manifests import create_split_manifest
from clinical_re_bilingual.schema import Language

In [ ]:
language = Language.ENGLISH
raw_root = Path(f"../data/raw/{language.value}")
parsed_path = Path(f"../data/interim/{language.value}_all.tsv")
layout = I2B2Layout(
    concepts_dir=raw_root / "concepts",
    relations_dir=raw_root / "relations",
    text_dir=raw_root / "text",
)
parsed = parse_i2b2_directory(layout, language)
write_tsv(parsed, parsed_path)
len(parsed)

After constructing the authorized 1,500-example training pool and untouched 500-example official test TSVs, create the public-safe manifest.

In [ ]:
training_path = Path(f"../data/processed/{language.value}_training_pool.tsv")
test_path = Path(f"../data/processed/{language.value}_official_test.tsv")
manifest = create_split_manifest(
    read_relation_examples_tsv(training_path),
    read_relation_examples_tsv(test_path),
    dataset_name=f"i2b2-2010-{language.value}",
    dev_fraction=0.2,
    seed=42,
)
manifest.write_json(Path(f"../data/manifests/{language.value}_split.json"))
manifest.label_counts